# Render Sentinel-2 chips into labelling PNGs.

Each output PNG shows three panels side by side:
    RGB (true colour)  |  NDVI (vegetation)  |  MNDWI (water)
with a colour bar on each index panel.

These PNGs are a human cross-check only. The labelling tool (wh_label.py) draws
its panels live from the GeoTIFF, and labels are always painted onto the raster
grid — never onto anything rendered here.

Two deliberate scaling choices, both so the SAME waterhole looks consistent from
month to month (important for temporal labelling):
  * RGB uses a FIXED reflectance stretch, NOT a per-band percentile stretch.
    Per-band stretching rescales each channel to its own min/max, which breaks
    the true R:G:B balance (magenta/lilac casts), amplifies composite noise into
    speckle, and makes every chip scale differently.
  * The index panels use TIGHT fixed display ranges centred on where the data
    actually sits, so contrast lands on the scene instead of being spent on
    [-1, 1] values that never occur.

Changed for the re-exported chips (cookie-cutting/images_tif_v2):
  * Reads from images_tif_v2: 13 bands, EPSG:32753, 10 m square pixels.
  * Nodata is now EXPLICIT. Gaps are -9999 (no clear observation in that month)
    or -inf (thin border where the reprojected AOI does not fill the raster).
    Both become NaN here and are drawn in a flat colour, so a gap can never be
    mistaken for dark water. The previous version inferred gaps from
    'both bands are exactly zero', which missed pixels where only ONE band was
    zero and rendered them as index values of exactly +/-1.
  * The n_obs band is reported in each title: a median over one surviving scene
    is a different measurement from a median over six.
  * MNDWI is named MNDWI. The previous version computed MNDWI but labelled it
    NDWI everywhere.
  * NDVI display range tightened; -0.5 to 0.8 spent almost all its contrast on
    values that do not occur, leaving dry-season panels a flat wash.

There are 187 sites x 84 months = 15,708 chips. Rendering all of them takes
hours and is rarely what you want, so SITE_FILTER / MONTH_FILTER below restrict
the run to the chips you are about to label.



In [ ]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")            # render straight to file, no GUI window
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import rasterio

## Load modules specific to this script

In [ ]:
# wh_* modules live in cookie-cutting/ alongside the chips
COOKIE_CUTTING = Path("cookie-cutting").resolve()
if str(COOKIE_CUTTING) not in sys.path:
    sys.path.insert(0, str(COOKIE_CUTTING))

import wh_config
import wh_indices
import wh_naming

cfg = wh_config.load()

## Set parameters

Some are drawn from the yaml file at `cookie-cutting/waterhole_seg_config.yaml`

In [ ]:
# --- what to render ------------------------------------------------------
input_tif_dir = cfg.paths["tiles"]
output_png_dir = cfg.paths["chips_png"]
output_png_dir.mkdir(parents=True, exist_ok=True)

# Restrict the run. None means "all", which is 15,708 chips.
SITE_FILTER = None            # e.g. {"025", "075", "100"}
MONTH_FILTER = None           # e.g. {"2024-03", "2024-10"}
OVERWRITE = False             # existing PNGs are skipped unless True

# --- display settings ----------------------------------------------------
RGB_BANDS = ("B4", "B3", "B2")
RGB_MAX_REFLECTANCE = 0.30    # lower for a brighter image; raise if sand clips
RGB_GAMMA = 0.85              # <1 brightens midtones for dark tropical scenes

NDVI_CMAP = "YlGn"
NDVI_RANGE = (-0.1, 0.8)

MNDWI_CMAP = "RdBu"           # diverging: dry land -> red/brown, water -> blue
MNDWI_VMIN, MNDWI_VCENTER, MNDWI_VMAX = -0.8, 0.0, 0.6

GAP_COLOUR = "#ff00ff"        # magenta: unmistakably "no data", never a surface

## Helper functions

In [ ]:
def true_colour(bands):
    """Natural-colour RGB on a fixed reflectance stretch shared by all channels.

    Returns an (H, W, 3) float array in [0, 1]. NaN (gap) pixels come back as
    NaN and are painted by the caller.
    """
    stack = np.dstack([bands[name] for name in RGB_BANDS]).astype("float32")
    stack = np.clip(stack / RGB_MAX_REFLECTANCE, 0, 1)
    return np.power(stack, RGB_GAMMA)


def paint_gaps(axis, gap_mask):
    """Overlay the nodata mask in a flat colour on top of whatever was drawn."""
    overlay = np.zeros(gap_mask.shape + (4,), dtype=float)
    overlay[gap_mask] = mcolors.to_rgba(GAP_COLOUR)
    axis.imshow(overlay, interpolation="nearest")

In [ ]:
band_names = list(cfg["tiles"]["bands"])
obs_band = cfg["tiles"]["obs_band"]
nodata_value = float(cfg["tiles"]["nodata"])

paths = sorted(input_tif_dir.glob("*.tif"))
if SITE_FILTER or MONTH_FILTER:
    selected = []
    for path in paths:
        key = wh_naming.parse_path(path)
        if SITE_FILTER and key.site_id not in SITE_FILTER:
            continue
        if MONTH_FILTER and key.year_month not in MONTH_FILTER:
            continue
        selected.append(path)
    paths = selected

print(f"{len(paths):,} chip(s) selected from {input_tif_dir}")

n_rendered = 0
n_skipped = 0

for tif_path in paths:
    output_path = output_png_dir / f"{tif_path.stem}.png"
    if output_path.exists() and not OVERWRITE:
        n_skipped += 1
        continue

    key = wh_naming.parse_path(tif_path)

    with rasterio.open(tif_path) as dataset:
        raw = dataset.read().astype("float32")

    optical = raw[: len(band_names)]
    # Both nodata conventions in one rule. -inf is not equal to -9999, so
    # testing the tag alone would leave the border strip looking like real data.
    gap = (optical == nodata_value) | ~np.isfinite(optical)
    optical = np.where(gap, np.nan, optical)
    bands = {name: optical[i] for i, name in enumerate(band_names)}

    gap_any = np.any(gap, axis=0)

    n_obs = raw[len(band_names)] if raw.shape[0] > len(band_names) else None
    if n_obs is not None:
        observed = np.isfinite(n_obs) & (n_obs > 0)
        mean_obs = float(n_obs[observed].mean()) if observed.any() else 0.0
        obs_note = f"n_obs mean {mean_obs:.1f}, gaps {100 * gap_any.mean():.0f}%"
    else:
        obs_note = f"gaps {100 * gap_any.mean():.0f}%"

    ndvi = wh_indices.compute("ndvi", bands)
    mndwi = wh_indices.compute("mndwi", bands)

    figure, (rgb_axis, ndvi_axis, mndwi_axis) = plt.subplots(
        1, 3, figsize=(12, 4.5), constrained_layout=True
    )
    figure.suptitle(
        f"site {key.site_id}  {key.year_month}   ({obs_note})", fontsize=10
    )

    rgb_axis.imshow(true_colour(bands))
    rgb_axis.set_title("RGB (true colour)")

    ndvi_image = ndvi_axis.imshow(
        ndvi, cmap=NDVI_CMAP, vmin=NDVI_RANGE[0], vmax=NDVI_RANGE[1]
    )
    ndvi_axis.set_title("NDVI (vegetation)")
    figure.colorbar(ndvi_image, ax=ndvi_axis, fraction=0.046, pad=0.04, label="NDVI")

    mndwi_norm = mcolors.TwoSlopeNorm(
        vmin=MNDWI_VMIN, vcenter=MNDWI_VCENTER, vmax=MNDWI_VMAX
    )
    mndwi_image = mndwi_axis.imshow(mndwi, cmap=MNDWI_CMAP, norm=mndwi_norm)
    mndwi_axis.set_title("MNDWI (water)")
    figure.colorbar(mndwi_image, ax=mndwi_axis, fraction=0.046, pad=0.04, label="MNDWI")

    for axis in (rgb_axis, ndvi_axis, mndwi_axis):
        paint_gaps(axis, gap_any)
        axis.set_xticks([])
        axis.set_yticks([])

    figure.savefig(output_path, dpi=120)
    plt.close(figure)
    n_rendered += 1

    if n_rendered % 50 == 0:
        print(f"  rendered {n_rendered:,}...")

print(f"rendered {n_rendered:,}, skipped {n_skipped:,} that already existed")